# Task 4 — Data Transformation and Storage

**COMP5339 Data Engineering · Assignment 1 · EV Charger Data Integration & Augmentation**

---

This notebook turns the cleaned charger data from Task 2 and the augmented attributes from Task 3
into a relational database in **DuckDB with the spatial extension**. It creates the schema from a
standalone DDL script, loads every table from the upstream outputs, checks that the load is complete
and correct, runs the kinds of analytical and spatial queries the design is meant to support, and
generates the schema diagram.

| Output | Contents |
|---|---|
| `sql/schema.sql` | DDL that recreates the whole schema: 13 tables with their constraints, 2 R-tree indexes, a distance macro and 3 analysis views |
| `data/processed/ev_chargers.duckdb` | The populated database |
| `docs/schema_diagram.png` | Entity-relationship diagram, generated from the database catalog |

### How to run

Run `01_data_acquisition.ipynb`, `02_data_cleaning.ipynb` and `03_data_augmentation.ipynb` first. Section 1.2
checks for their outputs and names the notebook to run if any are missing. Then run this notebook
from top to bottom (**Kernel → Restart Kernel and Run All Cells**). A run takes about 20 seconds.
It needs no network access, except to download the DuckDB spatial extension if Tasks 1–3 have not
already installed it.


### Contents

1. [Setup and configuration](#setup)
2. [Schema design](#design) - the decisions and the reasons for them
3. [Create the database](#create) from the DDL script
4. [Stage the upstream outputs](#stage)
5. [Load the tables](#load)
6. [Data Analysis and Observations](#6-data-analysis-and-observations)
8. [Schema diagram](#diagram)
9. [Summary and outputs](#summary)

<a id="setup"></a>
## 1. Setup and configuration

### 1.1 Dependencies

As in Tasks 1 and 2, the packages are installed from within the notebook so it runs on a fresh
machine. DuckDB must be **1.5 or newer**: the schema stores the coordinate reference system in the
geometry column type, which earlier versions do not support.

In [1]:
%pip install -q "duckdb>=1.5" "pandas>=2.2" "numpy>=1.26" "matplotlib>=3.8"

Note: you may need to restart the kernel to use updated packages.


In [9]:
import json
import re
import sys
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from IPython.display import Image, display

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

print(f"duckdb {duckdb.__version__} | pandas {pd.__version__}")

duckdb 1.5.5 | pandas 2.3.3


### 1.2 Configuration and inputs

Task 4 depends on the *outputs* of Tasks 1–3, never on their in-memory variables. Every input is
listed here together with the notebook that produces it. If any is missing, the cell stops with a
message saying which notebook to run, instead of failing later halfway through a load.

In [17]:
# --- Project folders ----------------------------------------------------------

PROJECT_ROOT = Path.cwd()

# Allow the notebook to run from either the project root or its parent folder.
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT / "COMP5339"

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"

# --- Task 4 outputs -----------------------------------------------------------

DDL_PATH = PROJECT_ROOT / "sql" / "schema.sql"
DB_PATH = PROCESSED_DIR / "ev_chargers.duckdb"
DIAGRAM_PATH = PROJECT_ROOT / "docs" / "schema_diagram.png"

# --- Required upstream inputs -------------------------------------------------
#
# The core schema needs:
#   1. the final cleaned and augmented charger records;
#   2. the normalized power-rating components;
#   3. the Task 1 manifest, which records the ABS shapefile path.

INPUTS = {
    "chargers": (
        PROCESSED_DIR / "ev_chargers_augmented.csv",
        "03_data_augmentation.ipynb",
    ),
    "power_ratings": (
        PROCESSED_DIR / "charger_power_ratings.csv",
        "03_data_augmentation.ipynb",
    ),
    "manifest": (
        RAW_DIR / "manifest.json",
        "01_data_acquisition.ipynb",
    ),
}

# Stop early and identify the notebook that produces each missing input.

missing = [
    f"  {path.relative_to(PROJECT_ROOT)} <- produced by {producer}"
    for path, producer in INPUTS.values()
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "Missing upstream outputs; run these notebooks first:\n"
        + "\n".join(missing)
    )

# Provide a simpler path-only lookup for later sections.

PATHS = {
    name: path
    for name, (path, _) in INPUTS.items()
}

# --- ABS SA4 shapefile --------------------------------------------------------

manifest_document = json.loads(
    PATHS["manifest"].read_text(encoding="utf-8")
)
artefacts = manifest_document["artefacts"]

SA4_SHAPEFILE_PATH = (
    PROJECT_ROOT
    / artefacts["abs_asgs_sa4_boundaries"]["shapefile"]
)

if not SA4_SHAPEFILE_PATH.exists():
    raise FileNotFoundError(
        f"{SA4_SHAPEFILE_PATH.relative_to(PROJECT_ROOT)} is missing; "
        "run 01_data_acquisition.ipynb."
    )

# --- Spatial configuration ---------------------------------------------------

SOURCE_CRS = "EPSG:7844"      # GDA2020 used by the ABS SA4 shapefile
TARGET_CRS = "EPSG:4326"      # WGS84 used by the charger coordinates
NSW_STATE_NAME = "New South Wales"


#output folders 
DDL_PATH.parent.mkdir(parents=True, exist_ok=True)
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
DIAGRAM_PATH.parent.mkdir(parents=True, exist_ok=True)

# --- Configuration summary ---------------------------------------------------

print("Required inputs:")
for name, path in PATHS.items():
    print(f"  {name:<15} {path.relative_to(PROJECT_ROOT)}")

print(f"  {'sa4_shapefile':<15} "
      f"{SA4_SHAPEFILE_PATH.relative_to(PROJECT_ROOT)}")

print("\nTask 4 outputs:")
print(f"  {'ddl':<15} {DDL_PATH.relative_to(PROJECT_ROOT)}")
print(f"  {'database':<15} {DB_PATH.relative_to(PROJECT_ROOT)}")
print(f"  {'diagram':<15} {DIAGRAM_PATH.relative_to(PROJECT_ROOT)}")

Required inputs:
  chargers        data\processed\ev_chargers_augmented.csv
  power_ratings   data\processed\charger_power_ratings.csv
  manifest        data\raw\manifest.json
  sa4_shapefile   data\raw\abs\sa4_shapefile\SA4_2026_AUST_GDA2020.shp

Task 4 outputs:
  ddl             sql\schema.sql
  database        data\processed\ev_chargers.duckdb
  diagram         docs\schema_diagram.png


<a id="design"></a>
## 2. Schema design

Task 3 produces a flat file with one row per cleaned charger record. That format is convenient for
exchange, but it repeats location, operator and region facts and stores multiple connector types in one
text field. Task 4 separates those facts into a normalised relational and spatial schema.

The design is a **normalised spatial relational schema based on Third Normal Form (3NF)
principles**. Each business fact is stored with the key it describes: site attributes depend on
`site_id`, operator attributes on `operator_id`, and charger attributes on `charger_id`. Child and
bridge tables represent repeating power components and multiple connector types without storing lists
in a single column. This reduces duplicated data, prevents inconsistent updates and allows primary
and foreign keys to enforce the relationships.

It is also spatial because charger sites are stored as points and SA4 regions as polygons. R-tree
indexes support efficient point-in-polygon, bounding-box and nearby-site queries. This combination
supports ordinary relational analysis and geographic analysis in the same DuckDB database.

### 2.1 Schema overview


| Table | What one row represents |
|---|---|
| `sa4_region` | One spatial NSW SA4 region from the ABS shapefile |
| `charging_site` | One coordinate-derived physical charging location |
| `operator` | One standardised charging operator |
| `charger` | One cleaned TfNSW charger or installation record |
| `charger_power_rating` | One power component of a charger rating |
| `connector_type` | One standard connector type, such as CCS2 or CHAdeMO |
| `charger_connector` | One supported connector type for one charger record |

### 2.2 Why the tables are separated

Location attributes belong to `charging_site`; equipment attributes belong to `charger`. One site
may contain several charger records, including separate AC and DC installations. `charger_id`
identifies a cleaned TfNSW record and does not necessarily represent one physical machine.

`operator` stores each canonical company name and website once. `charger_power_rating` preserves
compound ratings: `2x350kW & 2x175kW` becomes two rows instead of one incomplete maximum value.

A charger can support several connector types, and the same connector type can occur on many chargers.
`connector_type` and `charger_connector` represent this many-to-many relationship without storing a list
such as `CCS2; CHAdeMO` in one database cell.

### 2.3 Spatial storage and indexing

The ABS shapefile uses GDA2020 (`EPSG:7844`), while charger coordinates use WGS84
(`EPSG:4326`). Every NSW SA4 boundary is transformed to WGS84 before insertion so polygons and
charger points use the same coordinate reference system.

| Geometry column | Stored geometry | Spatial index |
|---|---|---|
| `sa4_region.geom` | Full ABS polygon or multipolygon | R-tree |
| `charging_site.geom` | `ST_Point(longitude, latitude)` | R-tree |

The R-tree indexes reduce the candidate geometries examined by point-in-polygon, bounding-box and
nearby-site queries. They complement ordinary indexes; they are specifically intended for spatial
predicates.

Latitude and longitude remain as numeric columns for CSV export and non-spatial tools. The database
also loads NSW SA4 regions that contain no chargers, allowing regional reports to return zero rather
than omit those regions. ABS non-spatial codes are reported but excluded because they have no
polygon.

Example index definitions:

```sql
CREATE INDEX idx_charging_site_geom
ON charging_site USING RTREE (geom);

CREATE INDEX idx_sa4_region_geom
ON sa4_region USING RTREE (geom);
```

### 2.4 Keys and integrity

Task 2's `site_id` and `charger_id` are retained for traceability. Records with the same cleaned
coordinate pair share a `site_id`; each reconciled source record has one `charger_id`. Lookup IDs
for operators and connector types are generated during the database build.

| Relationship | Foreign key |
|---|---|
| Site belongs to an SA4 | `charging_site.sa4_code` → `sa4_region.sa4_code` |
| Charger belongs to a site | `charger.site_id` → `charging_site.site_id` |
| Charger has an operator | `charger.operator_id` → `operator.operator_id` |
| Power component belongs to a charger | `charger_power_rating.charger_id` → `charger.charger_id` |
| Plug link belongs to a charger | `charger_connector.charger_id` → `charger.charger_id` |
| Plug link identifies a connector type | `charger_connector.connector_type_id` → `connector_type.connector_type_id` |

The child tables use compound primary keys:

- `charger_power_rating (charger_id, component_no)`;
- `charger_connector (charger_id, connector_type_id)`.

Primary keys, foreign keys, `NOT NULL`, `UNIQUE` and `CHECK` constraints reject invalid records.
Coordinates must be valid, counts cannot be negative, power values must be positive, lookup names
must be unique, and non-null operator websites must be complete HTTPS URLs.

### 2.5 Design decisions

| Decision | Choice and reason |
|---|---|
| Storage model | Normalised relational schema to avoid repeated site and operator facts |
| Site and charger | Separate tables because one site may contain several charger records |
| Site address | Stored once in `charging_site` because it describes the location |
| Site name | Omitted because it is frequently missing and is not part of the coordinate-derived key |
| Geographic coverage | All spatial NSW SA4 regions from the ABS shapefile |
| Geometry CRS | EPSG:4326 for both transformed polygons and charger points |
| Spatial performance | R-tree indexes on site points and SA4 polygons |
| Power ratings | Child rows preserve every component of a compound rating |
| Connector types | Lookup and bridge tables represent the many-to-many relationship |


<a id="create"></a>
## 3. Create the database

This section creates the seven-table schema defined in Section 2. The complete DDL is stored in
`sql/schema.sql`, so the database structure can be recreated without relying on notebook state.

### 3.1 Rebuild and open DuckDB

The database is a derived output. Each run deletes the previous database and any interrupted
write-ahead log before opening a new DuckDB connection. This prevents obsolete objects from an
earlier schema from remaining in the file.

Storage format v1.5.0 is used because it preserves the CRS declared on
`GEOMETRY('EPSG:4326')` columns. DuckDB Spatial is loaded before the schema because the DDL uses
geometry types and R-tree indexes.


In [18]:
# Close a connection left by an earlier notebook run before deleting its file.
if "con" in globals():
    try:
        con.close()
    except Exception:
        pass

# Remove the previous derived database and an interrupted write-ahead log.
for stale_path in [DB_PATH, DB_PATH.with_name(DB_PATH.name + ".wal")]:
    stale_path.unlink(missing_ok=True)

DB_PATH.parent.mkdir(parents=True, exist_ok=True)

DB_STORAGE_VERSION = "v1.5.0"
# Open a new database using a format that preserves geometry CRS metadata.
con = duckdb.connect(
    str(DB_PATH),
    config={"storage_compatibility_version": DB_STORAGE_VERSION},
)

# Geometry types, transformations and R-tree indexes require DuckDB Spatial.
try:
    con.execute("LOAD spatial")
except duckdb.Error:
    con.execute("INSTALL spatial")
    con.execute("LOAD spatial")

print(f"Opened new database: {DB_PATH.relative_to(PROJECT_ROOT)}")


Opened new database: data\processed\ev_chargers.duckdb


### 3.2 Apply and inspect `schema.sql`

The standalone DDL creates `sa4_region`, `charging_site`, `operator`, `charger`,
`charger_power_rating`, `connector_type` and
`charger_connector`. It also creates R-tree indexes on the SA4 polygons and charging-site points.

Section 3.1 already removes the old database, so the DDL does not repeat `DROP TABLE` statements.
Rerun Section 3.1 before rerunning this cell. The tables are empty after this step. Section 4
prepares the source data, and Section 5 loads the
tables in parent-to-child order so every foreign key can be checked during insertion.


In [19]:
if not DDL_PATH.exists():
    raise FileNotFoundError(f"Schema file is missing: {DDL_PATH}")


# Apply the complete standalone schema definition.
con.execute(DDL_PATH.read_text(encoding="utf-8"))

storage_version = con.execute("""
    SELECT tags['storage_version']
    FROM duckdb_databases()
    WHERE database_name = current_database()
""").fetchone()[0]

print(f"Applied {DDL_PATH.relative_to(PROJECT_ROOT)}")
print(f"DuckDB storage format: {storage_version}")

# Inspect the seven core tables.
display(con.sql("""
    SELECT table_name, column_count, comment
    FROM duckdb_tables()
    WHERE database_name = current_database()
    ORDER BY table_name
""").df())

# Inspect the two spatial indexes defined by the DDL.
display(con.sql("""
    SELECT index_name, table_name, is_unique, expressions
    FROM duckdb_indexes()
    WHERE database_name = current_database()
    ORDER BY table_name, index_name
""").df())


Applied sql\schema.sql
DuckDB storage format: v1.5.0+


,table_name,column_count,comment
0,charger,7,Cleaned TfNSW charger or installation records
1,charger_connector,2,Many-to-many relationship between chargers and connector...
2,charger_power_rating,4,Power components parsed from charger ratings
3,charging_site,8,Coordinate-derived physical EV charging locations
4,connector_type,2,Standard EV connector or plug types
5,operator,3,Canonical EV charging operators
6,sa4_region,7,Spatial NSW SA4 boundaries from the ABS shapefile


,index_name,table_name,is_unique,expressions
0,idx_charging_site_geom,charging_site,False,[geom]
1,idx_sa4_region_geom,sa4_region,False,[geom]


<a id="stage"></a>
## 4. Stage the upstream outputs

Task 4 reads only the two tabular outputs needed by the core schema. The final Task 3 file supplies
site, charger, operator, cost and connector attributes. Task 3 also supplies the Task 2 power
components filtered to its final retained charger IDs. The ABS shapefile is read directly in
Section 5.

### 4.1 Read the charger and power files

Identifiers, postcodes and ABS codes receive explicit pandas types. This prevents identifiers with
missing values becoming floating-point numbers and preserves leading zeroes in text codes. The
`connector_type` column is retained as text here; Section 5 separates values such as
`CCS2; CHAdeMO` into lookup and bridge rows.


In [20]:
charger_dtypes = {
    "charger_id": "Int64",
    "site_id": "Int64",
    "postcode": "string",
    "sa4_code": "string",
    "number_of_plugs": "Int64",
    "connector_type": "string",
    "cost_applies": "boolean",
}

power_dtypes = {
    "charger_id": "Int64",
    "component_no": "Int64",
    "connector_count": "Int64",
    "power_kw": "float64",
}

chargers = pd.read_csv(PATHS["chargers"], dtype=charger_dtypes)
power_ratings = pd.read_csv(PATHS["power_ratings"], dtype=power_dtypes)

print(
    f"Loaded {len(chargers):,} charger records from "
    f"{PATHS['chargers'].relative_to(PROJECT_ROOT)}"
)
print(
    f"Loaded {len(power_ratings):,} power components from "
    f"{PATHS['power_ratings'].relative_to(PROJECT_ROOT)}"
)


Loaded 1,926 charger records from data\processed\ev_chargers_augmented.csv
Loaded 1,507 power components from data\processed\charger_power_ratings.csv


### 4.2 Validate and register staging views

Validation happens before any row is inserted. The checks confirm that charger IDs are unique,
site coordinates are usable, each coordinate-derived site has one coordinate and SA4 assignment,
power-component keys are unique, and every staged power component references a retained charger.

Task 3 has already filtered the power-component table to the retained charger IDs. This section
therefore validates referential completeness without changing or removing any input rows.

The DataFrames are registered as temporary DuckDB views. Registration does not copy them into the
database; Section 5 performs the controlled transformations and insertions.


In [ ]:
required_charger_columns = {
    "charger_id", "site_id", "station_address", "postcode", "lga_name",
    "latitude", "longitude", "sa4_code", "operator", "operator_website",
    "charger_type", "charger_status", "number_of_plugs", "cost_applies",
    "connector_type",
}
required_power_columns = {
    "charger_id", "component_no", "connector_count", "power_kw",
}

missing_charger_columns = sorted(required_charger_columns - set(chargers.columns))
missing_power_columns = sorted(required_power_columns - set(power_ratings.columns))

if missing_charger_columns:
    raise ValueError(
        "Task 3 output is missing columns: " + ", ".join(missing_charger_columns)
    )
if missing_power_columns:
    raise ValueError(
        "Power-rating output is missing columns: " + ", ".join(missing_power_columns)
    )

assert chargers["charger_id"].notna().all()
assert chargers["charger_id"].is_unique
assert chargers["site_id"].notna().all()
assert chargers[["latitude", "longitude", "sa4_code"]].notna().all().all()
assert chargers["latitude"].between(-90, 90).all()
assert chargers["longitude"].between(-180, 180).all()

# A coordinate-derived site must have one coordinate and one SA4 assignment.
site_consistency = chargers.groupby("site_id")[
    ["latitude", "longitude", "sa4_code"]
].nunique(dropna=False)
assert site_consistency.le(1).all().all()

retained_charger_ids = set(chargers["charger_id"].astype(int))
assert not power_ratings.duplicated(["charger_id", "component_no"]).any()
assert power_ratings["connector_count"].gt(0).all()
assert power_ratings["power_kw"].gt(0).all()
assert set(power_ratings["charger_id"].astype(int)) <= retained_charger_ids

#temporary registered view
con.register("stg_charger", chargers)
con.register("stg_power_rating", power_ratings)

display(pd.DataFrame(
    [
        {
            "staging_view": "stg_charger",
            "source": str(PATHS["chargers"].relative_to(PROJECT_ROOT)),
            "rows": len(chargers),
            "columns": chargers.shape[1],
        },
        {
            "staging_view": "stg_power_rating",
            "source": str(PATHS["power_ratings"].relative_to(PROJECT_ROOT)),
            "rows": len(power_ratings),
            "columns": power_ratings.shape[1],
        },
    ]
))

print(
    f"Validated {chargers.site_id.nunique():,} sites and "
    f"{chargers.charger_id.nunique():,} charger records."
)


,staging_view,source,rows,columns
0,stg_charger,data\processed\ev_chargers_augmented.csv,1926,24
1,stg_power_rating,data\processed\charger_power_ratings.csv,1507,4


Validated 1,915 sites and 1,926 charger records.


<a id="load"></a>
## 5. Load the tables

Tables are loaded in parent-to-child order so DuckDB can enforce every foreign key during
insertion. Geographic regions are loaded first because every charging site references an SA4.

### 5.1 Load spatial SA4 regions

DuckDB Spatial reads the original ABS shapefile directly with `ST_Read`. The staging table keeps
all NSW SA4 records so records without geometry remain visible for audit. Only rows containing a
real polygon are inserted into `sa4_region`; ABS non-spatial codes cannot participate in spatial
queries and are reported separately.

The shapefile uses GDA2020 (`EPSG:7844`). Each retained boundary is transformed to WGS84
(`EPSG:4326`) before insertion, matching the coordinate system used by charging-site points. The
table includes every spatial NSW SA4 region, even when it contains no chargers.


In [22]:
# Read all NSW SA4 records from the original ABS shapefile.
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE stg_sa4 AS
    SELECT
        CAST(SA4_CODE26 AS VARCHAR) AS sa4_code,
        SA4_NAME26 AS sa4_name,
        CAST(GCC_CODE26 AS VARCHAR) AS gcc_code,
        GCC_NAME26 AS gcc_name,
        STE_NAME26 AS state_name,
        AREASQKM26 AS area_sqkm,
        geom
    FROM ST_Read('{SA4_SHAPEFILE_PATH.as_posix()}')
    WHERE STE_NAME26 = '{NSW_STATE_NAME}'
""")

# Non-spatial ABS categories have no polygon and cannot be inserted.
non_spatial_sa4 = con.sql("""
    SELECT sa4_code, sa4_name, area_sqkm
    FROM stg_sa4
    WHERE geom IS NULL OR ST_IsEmpty(geom)
    ORDER BY sa4_code
""").df()

print(f"Non-spatial NSW SA4 records excluded: {len(non_spatial_sa4)}")
display(non_spatial_sa4)

# Transform every real NSW boundary from GDA2020 to WGS84 and store it.
con.execute(f"""
    INSERT INTO sa4_region (
        sa4_code,
        sa4_name,
        gcc_code,
        gcc_name,
        state_name,
        area_sqkm,
        geom
    )
    SELECT
        sa4_code,
        sa4_name,
        gcc_code,
        gcc_name,
        state_name,
        area_sqkm,
        ST_Transform(
            geom,
            '{SOURCE_CRS}',
            '{TARGET_CRS}',
            always_xy := true
        )
    FROM stg_sa4
    WHERE geom IS NOT NULL AND NOT ST_IsEmpty(geom)
    ORDER BY sa4_code
""")

source_spatial_count = con.sql("""
    SELECT COUNT(*)
    FROM stg_sa4
    WHERE geom IS NOT NULL AND NOT ST_IsEmpty(geom)
""").fetchone()[0]

loaded_region_count = con.sql(
    "SELECT COUNT(*) FROM sa4_region"
).fetchone()[0]

assert loaded_region_count == source_spatial_count

# Confirm that every stored boundary is present, non-empty and spatially valid.
region_validation = con.sql("""
    SELECT
        COUNT(*) AS sa4_regions,
        COUNT(DISTINCT sa4_code) AS unique_codes,
        COUNT(*) FILTER (WHERE geom IS NULL OR ST_IsEmpty(geom)) AS empty_geometries,
        COUNT(*) FILTER (WHERE NOT ST_IsValid(geom)) AS invalid_geometries,
        CAST(SUM(ST_NPoints(geom)) AS BIGINT) AS boundary_vertices
    FROM sa4_region
""").df()

assert region_validation.loc[0, "sa4_regions"] == region_validation.loc[0, "unique_codes"]
assert region_validation.loc[0, "empty_geometries"] == 0
assert region_validation.loc[0, "invalid_geometries"] == 0

print(
    f"Loaded {loaded_region_count} spatial NSW SA4 regions "
    f"in {TARGET_CRS}."
)
display(region_validation)


Non-spatial NSW SA4 records excluded: 2


,sa4_code,sa4_name,area_sqkm
0,197,Migratory - Offshore - Shipping (NSW),NaN
1,199,No usual address (NSW),NaN


Loaded 28 spatial NSW SA4 regions in EPSG:4326.


,sa4_regions,unique_codes,empty_geometries,invalid_geometries,boundary_vertices
0,28,28,0,0,755051


### 5.2 Load operators

Task 2 standardised operator aliases, and Task 3 standardised accepted operator websites to HTTPS
domain roots. The `operator` table stores each canonical operator once instead of repeating its name
and website on every charger record.

An operator can appear on many staged rows. If its non-null website values disagree, the most
frequent website is selected; an alphabetical tie-break makes repeated builds deterministic.
Operators with no known website remain in the table with a null value. Missing operator names are
not converted into a false `Unknown` company; their chargers retain a null foreign key.


In [23]:
# Show operators that have more than one candidate website before consolidation.
operator_website_conflicts = con.sql("""
    SELECT
        operator AS operator_name,
        COUNT(DISTINCT operator_website) AS website_count,
        STRING_AGG(
            DISTINCT operator_website,
            '; ' ORDER BY operator_website
        ) AS candidate_websites
    FROM stg_charger
    WHERE operator IS NOT NULL
      AND TRIM(operator) <> ''
      AND operator_website IS NOT NULL
      AND TRIM(operator_website) <> ''
    GROUP BY operator
    HAVING COUNT(DISTINCT operator_website) > 1
    ORDER BY operator
""").df()

print(f"Operators with conflicting website values: {len(operator_website_conflicts)}")
display(operator_website_conflicts)


Operators with conflicting website values: 0


,operator_name,website_count,candidate_websites


In [24]:
#Insert to Operator table with a unique operator_id, operator_name, and the most common operator_website
con.execute("""
    WITH names AS (
        SELECT DISTINCT
            operator AS operator_name
        FROM stg_charger
        WHERE operator IS NOT NULL
          AND TRIM(operator) <> ''
    ),

    website_votes AS (
        SELECT
            operator AS operator_name,
            operator_website,
            COUNT(*) AS votes
        FROM stg_charger
        WHERE operator IS NOT NULL
          AND TRIM(operator) <> ''
          AND operator_website IS NOT NULL
          AND TRIM(operator_website) <> ''
        GROUP BY
            operator,
            operator_website
    ),

    preferred_website AS (
        SELECT
            operator_name,
            operator_website
        FROM website_votes
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY operator_name
            ORDER BY votes DESC, operator_website
        ) = 1
    )

    INSERT INTO operator (
        operator_id,
        operator_name,
        operator_website
    )
    SELECT
        ROW_NUMBER() OVER (
            ORDER BY
                LOWER(n.operator_name),
                n.operator_name
        ) AS operator_id,
        n.operator_name,
        w.operator_website
    FROM names AS n
    LEFT JOIN preferred_website AS w
        USING (operator_name)
    ORDER BY operator_id
""")

# Validate uniqueness and the website rule enforced by the schema.
operator_validation = con.sql("""
    SELECT
        COUNT(*) AS operators,
        COUNT(DISTINCT operator_name) AS unique_names,
        COUNT(operator_website) AS with_website,
        COUNT(*) FILTER (
            WHERE operator_website IS NOT NULL
              AND NOT starts_with(operator_website, 'https://')
        ) AS invalid_websites
    FROM operator
""").df()

assert operator_validation.loc[0, "operators"] == operator_validation.loc[0, "unique_names"]
assert operator_validation.loc[0, "invalid_websites"] == 0

display(operator_validation)
display(con.sql("""
    SELECT operator_id, operator_name, operator_website
    FROM operator
    ORDER BY operator_id
    LIMIT 10
""").df())


,operators,unique_names,with_website,invalid_websites
0,42,42,11,0


,operator_id,operator_name,operator_website
0,1,360 EV Charge,None
1,2,Alchemy Charge,None
2,3,Ampol,https://ampcharge.ampol.com.au
3,4,AXCharge,None
4,5,BMW,None
5,6,BP Australia,https://bp.com
6,7,CasaCharge,None
7,8,Charge OS,None
8,9,Chargefox,https://chargefox.com
9,10,ChargeHub,https://chargehub.solutions


### 5.4 Load charging sites

`site_id` was created in Task 2 from the cleaned coordinate pair. Several charger records can
therefore belong to one site. Coordinates, postcode and SA4 must agree within each `site_id` before
those attributes can move from the flat input into one `charging_site` row.



In [25]:
site_fields = [
    "latitude",
    "longitude",
    "postcode",
    "sa4_code",
    "station_address",
    "lga_name",
]

shared_site_ids = chargers.loc[
    chargers.duplicated("site_id", keep=False), "site_id"
].unique()

shared_sites = chargers.loc[
    chargers["site_id"].isin(shared_site_ids),
    ["site_id", "charger_id", *site_fields],
].copy()

site_variation = (
    shared_sites.groupby("site_id")[site_fields]
    .nunique(dropna=False)
    .gt(1)
    .sum()
    .rename("sites_where_value_varies")
    .to_frame()
)

print(
    f"{len(shared_site_ids)} sites contain more than one charger record "
    f"({len(shared_sites)} records)."
)
display(site_variation)

# These fields define the coordinate-derived site and must never disagree.
assert (site_variation.loc[
    ["latitude", "longitude", "postcode", "sa4_code"],
    "sites_where_value_varies",
] == 0).all()


10 sites contain more than one charger record (21 records).


,sites_where_value_varies
latitude,0
longitude,0
postcode,0
sa4_code,0
station_address,10
lga_name,4


#### Choose one canonical location row

Task 2 assigns the same `site_id` to charger records whose cleaned latitude and
longitude match exactly after rounding to six decimal places. Therefore, one
`site_id` represents one coordinate-based charging location, while several
charger records may belong to that location.

The site table requires only one row for each `site_id`. When several charger
records share the same coordinates, one source row is selected to provide the
site address, postcode and LGA. The selection prefers the row with the most
populated site fields, then the longer address, then the lower `charger_id` as
a deterministic tie-break.


The shared longitude and latitude are converted into an EPSG:4326 point. After
insertion, the code validates row counts, foreign keys, point coordinates and
geometry validity, then removes the temporary selection table.

In [26]:
# Rank source rows within each site and retain one canonical location record.
con.execute("""
    CREATE OR REPLACE TEMP TABLE stg_site_choice AS
    WITH ranked AS (
        SELECT
            *,
            (
                CASE WHEN station_address IS NOT NULL AND TRIM(station_address) <> '' THEN 1 ELSE 0 END
                + CASE WHEN postcode IS NOT NULL AND TRIM(postcode) <> '' THEN 1 ELSE 0 END
                + CASE WHEN lga_name IS NOT NULL AND TRIM(lga_name) <> '' THEN 1 ELSE 0 END
            ) AS site_completeness,
            ROW_NUMBER() OVER (
                PARTITION BY site_id
                ORDER BY
                    site_completeness DESC,
                    LENGTH(COALESCE(station_address, '')) DESC,
                    charger_id
            ) AS site_rank
        FROM stg_charger
    )
    SELECT *
    FROM ranked
    WHERE site_rank = 1
""")

con.execute("""
    INSERT INTO charging_site (
        site_id,
        station_address,
        postcode,
        lga_name,
        latitude,
        longitude,
        geom,
        sa4_code
    )
    SELECT
        site_id,
        station_address,
        postcode,
        lga_name,
        latitude,
        longitude,
        ST_Point(longitude, latitude)::GEOMETRY('EPSG:4326'),
        sa4_code
    FROM stg_site_choice
    ORDER BY site_id
""")

site_validation = con.sql("""
    SELECT
        COUNT(*) AS sites,
        COUNT(DISTINCT site_id) AS unique_site_ids,
        COUNT(*) FILTER (WHERE s.geom IS NULL OR ST_IsEmpty(s.geom)) AS empty_points,
        COUNT(*) FILTER (WHERE NOT ST_IsValid(s.geom)) AS invalid_points,
        COUNT(*) FILTER (
            WHERE ABS(ST_X(s.geom) - longitude) > 0.0000001
               OR ABS(ST_Y(s.geom) - latitude) > 0.0000001
        ) AS coordinate_mismatches,
        COUNT(*) FILTER (WHERE r.sa4_code IS NULL) AS missing_sa4_references
    FROM charging_site AS s
    LEFT JOIN sa4_region AS r USING (sa4_code)
""").df()

assert site_validation.loc[0, "sites"] == chargers["site_id"].nunique()
assert site_validation.loc[0, "sites"] == site_validation.loc[0, "unique_site_ids"]
assert site_validation.loc[0, "empty_points"] == 0
assert site_validation.loc[0, "invalid_points"] == 0
assert site_validation.loc[0, "coordinate_mismatches"] == 0
assert site_validation.loc[0, "missing_sa4_references"] == 0

display(site_validation)

print("Canonical choices for shared sites:")
display(con.sql("""
    SELECT
        s.site_id,
        s.charger_id AS selected_from_charger_id,
        s.station_address,
        s.lga_name,
        s.site_completeness
    FROM stg_site_choice AS s
    WHERE s.site_id IN (
        SELECT site_id
        FROM stg_charger
        GROUP BY site_id
        HAVING COUNT(*) > 1
    )
    ORDER BY s.site_id
""").df())

con.execute("DROP TABLE stg_site_choice")


,sites,unique_site_ids,empty_points,invalid_points,coordinate_mismatches,missing_sa4_references
0,1915,1915,0,0,0,0


Canonical choices for shared sites:


,site_id,selected_from_charger_id,station_address,lga_name,site_completeness
0,246,247,"19 Princes Hwy, Figtree NSW 2525, Australia",Wollongong City Council,3
1,479,481,"345 Victoria Ave, Chatswood NSW 2067",Willoughby City Council,3
2,483,485,"3-5 Underwood Rd, Homebush NSW 2140",Strathfield Municipal Council,3
3,543,547,42 Lake Canobolas Rd Nashdale NSW 2800 Australia,Cabonne Council,3
4,594,598,"52 Pembroke St, Ashfield NSW 2131, Australia",Inner West Council,3
5,699,704,74 Maitland St Bingara NSW 2404 Australia,Gwydir Shire Council,3
6,966,972,Early Start Discovery Space (Building 21) University of ...,Wollongong City Council,3
7,980,988,33 Gugandi Rd Narara NSW 2250 Australia,Central Coast Council,3
8,981,991,53 Railway St Griffith NSW 2680 Australia,Griffith City Council,3
9,1173,1184,"260 Victoria Avenue, Chatswood, NSW 2067",Willoughby City Council,3


### 5.5 Load chargers

`charger` stores one row for every final Task 3 charger record. Location attributes are already in
`charging_site`, while the operator name and website are represented by `operator_id`. The staged
operator name is therefore joined to the operator lookup before insertion.

Only charger-level facts remain: AC/DC type, operational status, reported plug count and whether a
cost applies. A `LEFT JOIN` preserves records whose operator is genuinely missing; every populated
operator name must resolve to exactly one lookup row.


In [27]:
# Confirm that every populated staged operator has a lookup-table match.
unmapped_operators = con.sql("""
    SELECT DISTINCT c.operator
    FROM stg_charger AS c
    LEFT JOIN operator AS o
        ON o.operator_name = c.operator
    WHERE c.operator IS NOT NULL
      AND TRIM(c.operator) <> ''
      AND o.operator_id IS NULL
    ORDER BY c.operator
""").df()

if not unmapped_operators.empty:
    display(unmapped_operators)
    raise ValueError("Some staged operators do not have an operator-table row")

# Insert every retained charger and replace its operator name with the foreign key.
con.execute("""
    INSERT INTO charger (
        charger_id,
        site_id,
        operator_id,
        charger_type,
        charger_status,
        number_of_plugs,
        cost_applies
    )
    SELECT
        c.charger_id,
        c.site_id,
        o.operator_id,
        c.charger_type,
        c.charger_status,
        c.number_of_plugs,
        c.cost_applies
    FROM stg_charger AS c
    LEFT JOIN operator AS o
        ON o.operator_name = c.operator
    ORDER BY c.charger_id
""")


In [28]:
charger_validation = con.sql("""
    SELECT
        COUNT(*) AS chargers,
        COUNT(DISTINCT charger_id) AS unique_charger_ids,
        COUNT(*) FILTER (WHERE s.site_id IS NULL) AS missing_site_references,
        COUNT(*) FILTER (
            WHERE c.operator_id IS NOT NULL AND o.operator_id IS NULL
        ) AS missing_operator_references,
        COUNT(*) FILTER (WHERE c.operator_id IS NULL) AS operator_unknown,
        COUNT(*) FILTER (WHERE c.cost_applies IS NOT NULL) AS cost_known
    FROM charger AS c
    LEFT JOIN charging_site AS s USING (site_id)
    LEFT JOIN operator AS o USING (operator_id)
""").df()

assert charger_validation.loc[0, "chargers"] == len(chargers)
assert charger_validation.loc[0, "chargers"] == charger_validation.loc[0, "unique_charger_ids"]
assert charger_validation.loc[0, "missing_site_references"] == 0
assert charger_validation.loc[0, "missing_operator_references"] == 0

display(charger_validation)

# Summarise the loaded charger characteristics without changing stored values.
display(con.sql("""
    SELECT
        COALESCE(charger_type, 'Not reported') AS charger_type,
        COALESCE(charger_status, 'Not reported') AS charger_status,
        COUNT(*) AS charger_records,
        CAST(SUM(number_of_plugs) AS BIGINT) AS reported_plugs,
        COUNT(*) FILTER (WHERE cost_applies = TRUE) AS cost_applies,
        COUNT(*) FILTER (WHERE cost_applies = FALSE) AS explicitly_free
    FROM charger
    GROUP BY charger_type, charger_status
    ORDER BY charger_type, charger_status
""").df())


,chargers,unique_charger_ids,missing_site_references,missing_operator_references,operator_unknown,cost_known
0,1926,1926,0,0,0,284


,charger_type,charger_status,charger_records,reported_plugs,cost_applies,explicitly_free
0,AC,Operational,1417,3407,0,0
1,DC,Operational,411,1584,269,15
2,Not reported,Upcoming,98,543,0,0


### 5.6 Load power ratings

Task 2 parsed each rating into one or more `(connector_count, power_kw)` components. Task 3 then
filtered that table to the charger records retained in its final output. Task 4 therefore loads
`stg_power_rating` directly and does not remove or rewrite any component.

The compound primary key `(charger_id, component_no)` preserves ratings such as
`2x350kW & 2x175kW` as two ordered rows. The foreign key ensures every component belongs to a
charger already loaded in Section 5.5.


In [29]:
# Load the already-filtered Task 3 power components without further removal.
con.execute("""
    INSERT INTO charger_power_rating (
        charger_id,
        component_no,
        connector_count,
        power_kw
    )
    SELECT
        charger_id,
        component_no,
        connector_count,
        power_kw
    FROM stg_power_rating
    ORDER BY charger_id, component_no
""")

power_validation = con.sql("""
    SELECT
        COUNT(*) AS components,
        COUNT(DISTINCT (charger_id, component_no)) AS unique_component_keys,
        COUNT(*) FILTER (WHERE c.charger_id IS NULL) AS missing_charger_references,
        COUNT(*) FILTER (WHERE connector_count <= 0) AS invalid_connector_counts,
        COUNT(*) FILTER (WHERE power_kw <= 0) AS invalid_power_values,
        COUNT(DISTINCT p.charger_id) AS chargers_with_rating,
        MIN(power_kw) AS minimum_kw,
        MAX(power_kw) AS maximum_kw
    FROM charger_power_rating AS p
    LEFT JOIN charger AS c USING (charger_id)
""").df()

assert power_validation.loc[0, "components"] == len(power_ratings)
assert power_validation.loc[0, "components"] == power_validation.loc[0, "unique_component_keys"]
assert power_validation.loc[0, "missing_charger_references"] == 0
assert power_validation.loc[0, "invalid_connector_counts"] == 0
assert power_validation.loc[0, "invalid_power_values"] == 0

multiple_component_chargers = con.sql("""
    SELECT COUNT(*)
    FROM (
        SELECT charger_id
        FROM charger_power_rating
        GROUP BY charger_id
        HAVING COUNT(*) > 1
    )
""").fetchone()[0]

display(power_validation)
print(f"Chargers with multiple power components: {multiple_component_chargers}")


,components,unique_component_keys,missing_charger_references,invalid_connector_counts,invalid_power_values,chargers_with_rating,minimum_kw,maximum_kw
0,1507,1507,0,0,0,1408,3.0,400.0


Chargers with multiple power components: 99


### 5.7 Load connector types and charger - connector relationships

Task 3 stores the standardized connector types for each matched DC charger in one text field, for
example `CCS2; CHAdeMO`. Task 4 separates that list into a lookup table and a bridge table so every
database cell contains one value.

`connector_type` stores each connector standard once. `charger_connector` represents the
many-to-many relationship: one charger can support several connector types, and one connector type
can be supported by many chargers. Missing connector information creates no bridge row and is not
converted into an `Unknown` category.


In [30]:
# Split each semicolon-separated value into one distinct charger/connector pair.
con.execute("""
    CREATE OR REPLACE TEMP TABLE stg_connector AS
    SELECT DISTINCT
        charger_id,
        TRIM(UNNEST(STRING_SPLIT(connector_type, ';'))) AS connector_name
    FROM stg_charger
    WHERE connector_type IS NOT NULL
      AND TRIM(connector_type) <> ''
""")

# Remove an empty token defensively if a future input contains a trailing separator.
con.execute("""
    DELETE FROM stg_connector
    WHERE connector_name IS NULL OR connector_name = ''
""")

# Assign reproducible lookup IDs in alphabetical order.
con.execute("""
    INSERT INTO connector_type (
        connector_type_id,
        connector_name
    )
    SELECT
        ROW_NUMBER() OVER (
            ORDER BY LOWER(connector_name), connector_name
        ) AS connector_type_id,
        connector_name
    FROM (
        SELECT DISTINCT connector_name
        FROM stg_connector
    )
    ORDER BY connector_type_id
""")

# Replace each connector name with its lookup-table foreign key.
con.execute("""
    INSERT INTO charger_connector (
        charger_id,
        connector_type_id
    )
    SELECT
        s.charger_id,
        c.connector_type_id
    FROM stg_connector AS s
    JOIN connector_type AS c USING (connector_name)
    ORDER BY s.charger_id, c.connector_type_id
""")

connector_validation = con.sql("""
    SELECT
        (SELECT COUNT(*) FROM connector_type) AS connector_types,
        (SELECT COUNT(DISTINCT connector_name) FROM connector_type) AS unique_names,
        COUNT(*) AS charger_connector_rows,
        COUNT(DISTINCT (cc.charger_id, cc.connector_type_id)) AS unique_links,
        COUNT(*) FILTER (WHERE ch.charger_id IS NULL) AS missing_charger_references,
        COUNT(*) FILTER (WHERE ct.connector_type_id IS NULL) AS missing_type_references
    FROM charger_connector AS cc
    LEFT JOIN charger AS ch USING (charger_id)
    LEFT JOIN connector_type AS ct USING (connector_type_id)
""").df()

assert connector_validation.loc[0, "connector_types"] == connector_validation.loc[0, "unique_names"]
assert connector_validation.loc[0, "charger_connector_rows"] == len(
    con.sql("SELECT * FROM stg_connector").df()
)
assert connector_validation.loc[0, "charger_connector_rows"] == connector_validation.loc[0, "unique_links"]
assert connector_validation.loc[0, "missing_charger_references"] == 0
assert connector_validation.loc[0, "missing_type_references"] == 0

display(connector_validation)
display(con.sql("""
    SELECT
        ct.connector_name,
        COUNT(*) AS chargers
    FROM charger_connector AS cc
    JOIN connector_type AS ct USING (connector_type_id)
    GROUP BY ct.connector_name
    ORDER BY chargers DESC, ct.connector_name
""").df())

con.execute("DROP TABLE stg_connector")


,connector_types,unique_names,charger_connector_rows,unique_links,missing_charger_references,missing_type_references
0,2,2,489,489,0,0


,connector_name,chargers
0,CCS2,315
1,CHAdeMO,174


## 6. Data analysis and observations

This section uses the normalised database to describe the stored NSW charging network. The queries
join sites, chargers, operators, regions, power ratings and connector types without returning to the
CSV files. Each result is followed by a short observation calculated from the current data, so the
commentary remains correct when the upstream dataset is refreshed.


### 6.1 Network overview

Count the main entities and compare AC and DC charger records. A site can contain more than one
charger, so charger and site counts answer different questions.


In [32]:
overview = con.sql("""
    SELECT
        COUNT(DISTINCT c.charger_id) AS chargers,
        COUNT(DISTINCT c.site_id) AS occupied_sites,
        COUNT(DISTINCT c.operator_id) AS operators,
        COUNT(DISTINCT s.sa4_code) AS sa4_regions_with_chargers,
        COUNT(*) FILTER (WHERE c.charger_type = 'DC') AS dc_chargers,
        COUNT(*) FILTER (WHERE c.charger_type = 'AC') AS ac_chargers
    FROM charger AS c
    JOIN charging_site AS s USING (site_id)
""").df()

display(overview)

row = overview.iloc[0]
dc_share = 100 * row.dc_chargers / row.chargers
print(
    f"Observation: {int(row.chargers):,} charger records occupy "
    f"{int(row.occupied_sites):,} sites. DC represents {dc_share:.1f}% of chargers."
)


,chargers,occupied_sites,operators,sa4_regions_with_chargers,dc_chargers,ac_chargers
0,1926,1915,42,28,411,1417


Observation: 1,926 charger records occupy 1,915 sites. DC represents 21.3% of chargers.


### 6.2 Largest DC charging operators

Rank operators by their DC charger records and report the number of physical sites and total plugs.
This separates network coverage from the amount of equipment recorded at those sites.


In [33]:
operator_summary = con.sql("""
    SELECT
        o.operator_name,
        COUNT(*) AS dc_chargers,
        COUNT(DISTINCT c.site_id) AS sites,
        SUM(c.number_of_plugs) AS recorded_plugs
    FROM charger AS c
    JOIN operator AS o USING (operator_id)
    WHERE c.charger_type = 'DC'
    GROUP BY o.operator_name
    ORDER BY dc_chargers DESC, sites DESC, o.operator_name
    LIMIT 10
""").df()

display(operator_summary)

leader = operator_summary.iloc[0]
print(
    f"Observation: {leader.operator_name} has the most DC records "
    f"({int(leader.dc_chargers):,}) across {int(leader.sites):,} sites."
)


,operator_name,dc_chargers,sites,recorded_plugs
0,Evie Networks,92,92,346.0
1,NRMA,65,65,190.0
2,Tesla,55,55,506.0
3,Chargefox,48,48,136.0
4,JOLT,47,47,91.0
5,Ampol,31,31,119.0
6,BP Australia,31,31,96.0
7,Exploren,23,23,50.0
8,Engie,6,6,12.0
9,Everty,4,4,8.0


Observation: Evie Networks has the most DC records (92) across 92 sites.


### 6.3 DC charger distribution by SA4

Count DC sites and chargers in every stored NSW SA4. The left joins retain regions with no matched
charging site, making gaps visible instead of silently excluding them.


In [34]:
sa4_summary = con.sql("""
    SELECT
        r.sa4_code,
        r.sa4_name,
        COUNT(DISTINCT s.site_id) FILTER (WHERE c.charger_type = 'DC') AS dc_sites,
        COUNT(c.charger_id) FILTER (WHERE c.charger_type = 'DC') AS dc_chargers
    FROM sa4_region AS r
    LEFT JOIN charging_site AS s USING (sa4_code)
    LEFT JOIN charger AS c USING (site_id)
    GROUP BY r.sa4_code, r.sa4_name
    ORDER BY dc_chargers DESC, r.sa4_name
""").df()

display(sa4_summary.head(10))

regions_without_dc = int((sa4_summary.dc_chargers == 0).sum())
leader = sa4_summary.iloc[0]
print(
    f"Observation: {leader.sa4_name} has the most DC chargers "
    f"({int(leader.dc_chargers):,}). {regions_without_dc} spatial NSW SA4 regions "
    "have no matched DC charger record."
)


,sa4_code,sa4_name,dc_sites,dc_chargers
0,121,Sydney - North Sydney and Hornsby,35,35
1,118,Sydney - Eastern Suburbs,30,30
2,101,Capital Region,27,27
3,122,Sydney - Northern Beaches,21,21
4,119,Sydney - Inner South West,19,19
5,117,Sydney - City and Inner South,18,18
6,103,Central West,17,17
7,107,Illawarra,16,17
8,120,Sydney - Inner West,17,17
9,116,Sydney - Blacktown,16,16


Observation: Sydney - North Sydney and Hornsby has the most DC chargers (35). 0 spatial NSW SA4 regions have no matched DC charger record.


### 6.4 Recorded DC power

Summarise the maximum recorded component power for each DC charger. Chargers without a numeric
rating remain in the denominator so the result also measures power-data completeness.


In [35]:
power_summary = con.sql("""
    WITH dc_power AS (
        SELECT
            c.charger_id,
            MAX(p.power_kw) AS maximum_power_kw
        FROM charger AS c
        LEFT JOIN charger_power_rating AS p USING (charger_id)
        WHERE c.charger_type = 'DC'
        GROUP BY c.charger_id
    )
    SELECT
        COUNT(*) AS dc_chargers,
        COUNT(maximum_power_kw) AS with_numeric_power,
        ROUND(100.0 * COUNT(maximum_power_kw) / COUNT(*), 1) AS coverage_percent,
        MEDIAN(maximum_power_kw) AS median_maximum_kw,
        MAX(maximum_power_kw) AS highest_kw,
        COUNT(*) FILTER (WHERE maximum_power_kw >= 50) AS at_least_50_kw,
        COUNT(*) FILTER (WHERE maximum_power_kw >= 150) AS at_least_150_kw
    FROM dc_power
""").df()

display(power_summary)

row = power_summary.iloc[0]
print(
    f"Observation: numeric power is available for {row.coverage_percent:.1f}% of DC records; "
    f"{int(row.at_least_150_kw):,} records reach at least 150 kW."
)


,dc_chargers,with_numeric_power,coverage_percent,median_maximum_kw,highest_kw,at_least_50_kw,at_least_150_kw
0,411,411,100.0,75.0,400.0,340,124


Observation: numeric power is available for 100.0% of DC records; 124 records reach at least 150 kW.


### 6.5 Connector-type augmentation

Measure Task 3 coverage and count each connector type. A charger supporting both CCS2 and CHAdeMO
appears once under each type, which is why connector counts should not be added together to obtain
the number of augmented chargers.


In [36]:
connector_coverage = con.sql("""
    SELECT
        COUNT(*) AS dc_chargers,
        COUNT(*) FILTER (
            WHERE EXISTS (
                SELECT 1
                FROM charger_connector AS cc
                WHERE cc.charger_id = c.charger_id
            )
        ) AS with_connector_type,
        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE EXISTS (
                    SELECT 1
                    FROM charger_connector AS cc
                    WHERE cc.charger_id = c.charger_id
                )
            ) / COUNT(*),
            1
        ) AS coverage_percent
    FROM charger AS c
    WHERE c.charger_type = 'DC'
""").df()

connector_counts = con.sql("""
    SELECT
        ct.connector_name,
        COUNT(DISTINCT cc.charger_id) AS dc_chargers
    FROM charger_connector AS cc
    JOIN connector_type AS ct USING (connector_type_id)
    JOIN charger AS c USING (charger_id)
    WHERE c.charger_type = 'DC'
    GROUP BY ct.connector_name
    ORDER BY dc_chargers DESC, ct.connector_name
""").df()

display(connector_coverage)
display(connector_counts)

row = connector_coverage.iloc[0]
leading_type = connector_counts.iloc[0]
print(
    f"Observation: connector type is known for {row.coverage_percent:.1f}% of DC records. "
    f"{leading_type.connector_name} is the most frequently recorded type "
    f"({int(leading_type.dc_chargers):,} chargers)."
)


,dc_chargers,with_connector_type,coverage_percent
0,411,315,76.6


,connector_name,dc_chargers
0,CCS2,315
1,CHAdeMO,174


Observation: connector type is known for 76.6% of DC records. CCS2 is the most frequently recorded type (315 chargers).


<a id="diagram"></a>
## 7. Schema diagram



<a id="summary"></a>
# 8. Summary and outputs

Checkpoint and close the database, then reopen it read-only. This confirms that the seven schema
tables and the EPSG:4326 geometry columns were saved correctly.


In [ ]:
con.execute("CHECKPOINT")
con.close()

expected_tables = {
    "sa4_region",
    "charging_site",
    "operator",
    "charger",
    "charger_power_rating",
    "connector_type",
    "charger_connector",
}

with duckdb.connect(str(DB_PATH), read_only=True) as reader:
    reader.execute("LOAD spatial")
    on_disk = reader.sql("""
        SELECT
            t.table_name,
            t.estimated_size AS rows,
            STRING_AGG(c.data_type, ', ')
                FILTER (WHERE c.data_type LIKE 'GEOMETRY%') AS geometry
        FROM duckdb_tables() AS t
        LEFT JOIN duckdb_columns() AS c
          ON c.database_name = t.database_name
         AND c.schema_name = t.schema_name
         AND c.table_name = t.table_name
        WHERE t.database_name = current_database()
          AND t.schema_name = 'main'
        GROUP BY t.table_name, t.estimated_size
        ORDER BY t.table_name
    """).df()

assert set(on_disk.table_name) == expected_tables
assert on_disk.geometry.dropna().str.contains("EPSG:4326").all()

print(
    f"{DB_PATH.relative_to(PROJECT_ROOT)}: "
    f"{DB_PATH.stat().st_size / 1e6:.1f} MB, {len(on_disk)} tables"
)
display(on_disk)


data\processed\ev_chargers.duckdb: 17.6 MB, 7 tables


,table_name,rows,geometry
0,charger,1926,None
1,charger_connector,489,None
2,charger_power_rating,1507,None
3,charging_site,1915,GEOMETRY('EPSG:4326')
4,connector_type,2,None
5,operator,42,None
6,sa4_region,28,GEOMETRY('EPSG:4326')


### What Task 4 produced

| File | Contents |
|---|---|
| `sql/schema.sql` | DDL for the seven-table relational and spatial schema |
| `data/processed/ev_chargers.duckdb` | Populated DuckDB database |
| `docs/schema_diagram.png` | Entity-relationship diagram |
| `tools/schema_diagram.py` | Code that generates the diagram from the database catalog |
